# CE807 Text Analytics Assignment — Student ID: 12345670

**Replace `12345670` with your actual 7/8-digit FASER number before submission.**

This notebook follows the required structure for CE807 (Spring 2026):

1. **Common Codes** — shared utilities, paths, metrics, I/O helpers  
2. **Method Unsupervised (LLM Prompting)** — `train_unsup` / `test_unsup`  
3. **Method Discriminative (Logistic Regression)** — `train_dis` / `test_dis`  
4. **Main Execution** — orchestrates training and testing end-to-end

Dataset: Amazon product reviews (Dataset ID = 71, binary sentiment).  
Labels: `positive` / `negative`.  
Output columns added to `test.csv`: `out_label_LR`, `out_label_Prompt_1` … `out_label_Prompt_5`.


## Install Required Libraries
All dependencies are installed automatically.

In [ ]:
# Install all required libraries automatically.
# PyTorch + HuggingFace as required by the assignment brief.
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("transformers>=4.38.0", "torch>=2.0.0", "sentencepiece", "accelerate")
_pip("scikit-learn>=1.3.0", "nltk", "gdown", "pandas", "numpy")
print("✓ All libraries installed.")


## Import Libraries

In [ ]:
# ── Standard library ──
import os, sys, pickle, random, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Numeric / Data ──
import numpy as np
import pandas as pd

# ── NLP preprocessing ──
import nltk
nltk.download("stopwords",  quiet=True)
nltk.download("wordnet",    quiet=True)
nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ── Scikit-learn ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.pipeline import Pipeline

# ── PyTorch + HuggingFace (as required) ──
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── File download (for model auto-download) ──
import gdown

print("✓ All imports successful.")


## Student ID & Dataset ID
Set these to your actual values.

In [ ]:
# ── IMPORTANT: replace with your real FASER number ──
STUDENT_ID = 12345670   # integer, used as random seed and folder name
DATA_ID    = 71         # dataset number assigned on Moodle


## Set Global Seeds
All seeds are initialised to `STUDENT_ID` for reproducibility.

In [ ]:
def set_global_seed(seed: int) -> None:
    """Set seeds for Python, NumPy, and PyTorch (CPU + CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    print(f"✓ Global seed set to {seed}")

set_global_seed(STUDENT_ID)


# Common Codes

This section contains all shared utilities:
- Path configuration
- Data reading
- Performance metrics
- Dataset statistics
- Model save / load (with Google Drive auto-download)


### Path Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Path Configuration
# Assignment spec: data at ./data/{DATA_ID}/, models at ./model/{STUDENT_ID}/
# Colab GDrive paths are also set if running inside Google Colab.
# ─────────────────────────────────────────────────────────────────────────────

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _in_colab()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive", force_remount=True)
    GDRIVE_ROOT = f"/content/gdrive/MyDrive/CE807-26-SP/Assignment"
    BASE_PATH   = GDRIVE_ROOT
else:
    BASE_PATH = "."   # run from directory containing ./data/ and ./model/

DATA_PATH       = os.path.join(BASE_PATH, "data",  str(DATA_ID))
MODEL_ROOT      = os.path.join(BASE_PATH, "model", str(STUDENT_ID))
MODEL_DIS_DIR   = os.path.join(MODEL_ROOT, "Model_Dis")
MODEL_UNSUP_DIR = os.path.join(MODEL_ROOT, "Model_Unsup")

os.makedirs(DATA_PATH,       exist_ok=True)
os.makedirs(MODEL_DIS_DIR,   exist_ok=True)
os.makedirs(MODEL_UNSUP_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_PATH, "train.csv")
VAL_FILE   = os.path.join(DATA_PATH, "valid.csv")
TEST_FILE  = os.path.join(DATA_PATH, "test.csv")

print(f"Base path      : {BASE_PATH}")
print(f"Data path      : {DATA_PATH}")
print(f"LR model dir   : {MODEL_DIS_DIR}")
print(f"LLM model dir  : {MODEL_UNSUP_DIR}")
print(f"Train file     : {TRAIN_FILE}")
print(f"Val file       : {VAL_FILE}")
print(f"Test file      : {TEST_FILE}")


### Data Reading

In [ ]:
def read_data(file_path: str) -> pd.DataFrame:
    """Read CSV and print basic info. Returns a DataFrame."""
    df = pd.read_csv(file_path)
    print(f"  Loaded  : {file_path}  →  {len(df):,} rows, {df.shape[1]} columns")
    return df


### Performance Metrics

In [ ]:
def compute_metrics(y_true, y_pred, label: str = "") -> dict:
    """
    Compute Accuracy, Macro-F1, Weighted-F1, and per-class F1.
    Prints a formatted report. Returns a metrics dictionary.
    """
    acc   = accuracy_score(y_true, y_pred)
    mf1   = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    wf1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    tag   = f"[{label}] " if label else ""

    print(f"\n{tag}{'─'*50}")
    print(f"  Accuracy        : {acc:.4f}")
    print(f"  Macro-F1        : {mf1:.4f}")
    print(f"  Weighted-F1     : {wf1:.4f}")
    print("\n" + classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:")
    print(cm)
    return {"accuracy": acc, "macro_f1": mf1, "weighted_f1": wf1}


### Dataset Statistics

In [ ]:
def print_dataset_statistics(df: pd.DataFrame, name: str = "Dataset") -> None:
    """Print label distribution and basic text-length stats."""
    print(f"\n{'='*55}")
    print(f"  {name}  —  {len(df):,} samples")
    print(f"{'='*55}")
    if "sentiment" in df.columns:
        vc = df["sentiment"].value_counts()
        for lbl, cnt in vc.items():
            pct = 100 * cnt / len(df)
            print(f"  {lbl:10s}: {cnt:5d}  ({pct:5.1f}%)")
    lengths = df["text"].str.len()
    print(f"  Text length — mean: {lengths.mean():.1f}, "
          f"median: {lengths.median():.1f}, max: {lengths.max()}")
    print()


### Model Save / Load with Auto-Download

In [ ]:
# ── Google Drive model URLs ────────────────────────────────────────────────
# After training, upload model files to Google Drive and paste shareable
# download links here (format: https://drive.google.com/uc?id=FILE_ID ).
# The auto-download runs automatically when local files are missing.
GDRIVE_URLS = {
    # Logistic Regression artefacts
    "lr_model.sav"       : "https://drive.google.com/uc?id=REPLACE_WITH_LR_MODEL_FILE_ID",
    "lr_vectorizer.sav"  : "https://drive.google.com/uc?id=REPLACE_WITH_LR_VECTORIZER_FILE_ID",
    "lr_config.sav"      : "https://drive.google.com/uc?id=REPLACE_WITH_LR_CONFIG_FILE_ID",
    # LLM best-model name (a small text file storing the model name string)
    "best_llm.txt"       : "https://drive.google.com/uc?id=REPLACE_WITH_BEST_LLM_FILE_ID",
}


def save_object(obj, path: str) -> None:
    """Pickle an object to disk."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  Saved  → {path}")


def load_object(path: str, gdrive_url: str = None):
    """
    Load a pickled object. If the file is missing and a Google Drive URL is
    provided, download it automatically (fulfils the assignment auto-download
    requirement).
    """
    fname = os.path.basename(path)
    if not os.path.exists(path):
        url = gdrive_url or GDRIVE_URLS.get(fname)
        if url and "REPLACE_WITH" not in url:
            print(f"  File not found locally. Downloading: {fname} …")
            os.makedirs(os.path.dirname(path), exist_ok=True)
            gdown.download(url, path, quiet=False)
        else:
            raise FileNotFoundError(
                f"Model file '{path}' not found. "
                "Please set the correct Google Drive URL in GDRIVE_URLS."
            )
    with open(path, "rb") as f:
        return pickle.load(f)


# Method Unsupervised — LLM Prompting

This section implements zero-shot / prompt-based sentiment classification using
open-source Seq2Seq language models from Hugging Face.

**Pipeline:**
1. Compare three LLMs (flan-t5-small / base / large) on a stratified validation sample.
2. Select the best-performing LLM.
3. Design and evaluate 5 distinct prompt strategies on the full validation set.
4. At test time, apply all 5 prompts and write `out_label_Prompt_1` … `out_label_Prompt_5`.


### Device & Model Setup

In [ ]:
def get_device() -> torch.device:
    """Return CUDA if available, else CPU."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Using device: {device}")
    return device


### Prompt Templates
Five diverse prompt strategies for binary sentiment classification.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Five Prompt Templates
# Each takes a `text` placeholder.
# ─────────────────────────────────────────────────────────────────────────────

PROMPTS = {
    # P1 — Direct / Minimal
    "Prompt_1": (
        "Classify the sentiment of the following product review. "
        "Reply with exactly one word: positive or negative.\n"
        "Review: {text}\n"
        "Sentiment:"
    ),

    # P2 — Explicit instruction with label definition
    "Prompt_2": (
        "You are a sentiment analysis system. "
        "A positive review expresses satisfaction; a negative review expresses dissatisfaction. "
        "Given the review below, output only 'positive' or 'negative'.\n"
        "Review: {text}\n"
        "Answer:"
    ),

    # P3 — Few-shot (2 examples per class)
    "Prompt_3": (
        "Determine whether each product review is positive or negative.\n\n"
        "Example 1: 'This product is amazing, works perfectly!' → positive\n"
        "Example 2: 'Broke after one day, total waste of money.' → negative\n"
        "Example 3: 'Great value, very happy with the purchase.' → positive\n"
        "Example 4: 'Poor quality, does not match description.' → negative\n\n"
        "Now classify:\n"
        "Review: {text}\n"
        "Sentiment:"
    ),

    # P4 — Role-based persona
    "Prompt_4": (
        "As an expert product reviewer, analyse the sentiment expressed in the "
        "following Amazon review. Provide a single-word answer: positive or negative.\n"
        "Review: {text}\n"
        "Sentiment:"
    ),

    # P5 — Chain-of-thought (brief reasoning then label)
    "Prompt_5": (
        "Read the product review and decide whether the customer is satisfied (positive) "
        "or dissatisfied (negative). Think step by step: identify key sentiment words, "
        "then give a one-word verdict.\n"
        "Review: {text}\n"
        "Verdict:"
    ),
}

print(f"✓ {len(PROMPTS)} prompt templates defined.")
for k in PROMPTS:
    print(f"  {k}: {PROMPTS[k][:60].strip()} …")


### Output Post-Processing

In [ ]:
def parse_sentiment(raw_output: str) -> str:
    """
    Convert raw model output to a clean 'positive' / 'negative' label.
    Strategy: look for 'negative' first (harder to false-match),
    then 'positive'; default to 'positive' if neither found.
    """
    text = raw_output.strip().lower()
    if "negative" in text or "neg" in text or "bad" in text or "poor" in text:
        return "negative"
    if "positive" in text or "pos" in text or "good" in text or "great" in text:
        return "positive"
    # For chain-of-thought, the final word is usually the verdict
    tokens = text.split()
    if tokens:
        last = tokens[-1].strip(".,!?")
        if "neg" in last:
            return "negative"
    return "positive"   # safe default (matches majority class)


### Batch Prediction Utility

In [ ]:
def batch_predict(model, tokenizer, device, texts: list,
                  prompt_template: str, batch_size: int = 16) -> list:
    """
    Run inference on `texts` using `prompt_template`.
    Returns a list of 'positive' / 'negative' labels.
    """
    model.eval()
    preds = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start: start + batch_size]
        prompts = [prompt_template.format(text=t) for t in batch_texts]
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(device)
        with torch.no_grad():
            out_ids = model.generate(
                **enc,
                max_new_tokens=10,
                num_beams=2,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        preds.extend([parse_sentiment(d) for d in decoded])
    return preds


### LLM Comparison Function
Compare flan-t5-small, flan-t5-base, and flan-t5-large on a stratified validation sample.

In [ ]:
CANDIDATE_LLMS = [
    "google/flan-t5-small",   # ~80M params  — fast
    "google/flan-t5-base",    # ~250M params — balanced
    "google/flan-t5-large",   # ~780M params — accurate
]


def compare_llms(val_df: pd.DataFrame, sample_size: int = 150,
                 prompt_key: str = "Prompt_1") -> str:
    """
    Evaluate each candidate LLM on a stratified sample of the validation set.
    Uses `prompt_key` (default Prompt_1 — direct) for a fair comparison.
    Returns the name of the best-performing LLM.
    """
    set_global_seed(STUDENT_ID)
    device = get_device()

    # Stratified sample (equal classes if possible)
    pos = val_df[val_df["sentiment"] == "positive"].sample(
        min(sample_size // 2, sum(val_df["sentiment"] == "positive")),
        random_state=STUDENT_ID)
    neg = val_df[val_df["sentiment"] == "negative"].sample(
        min(sample_size // 2, sum(val_df["sentiment"] == "negative")),
        random_state=STUDENT_ID)
    sample = pd.concat([pos, neg]).sample(frac=1, random_state=STUDENT_ID)
    y_true = sample["sentiment"].tolist()
    texts  = sample["text"].tolist()

    template = PROMPTS[prompt_key]
    results  = {}

    print(f"\n{'='*60}")
    print(f"  LLM Comparison  (sample={len(sample)}, prompt={prompt_key})")
    print(f"{'='*60}")

    for model_name in CANDIDATE_LLMS:
        print(f"\n  Loading {model_name} …")
        try:
            tok = AutoTokenizer.from_pretrained(model_name)
            mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
            bs  = 16 if device.type == "cuda" else 8
            preds = batch_predict(mdl, tok, device, texts, template, batch_size=bs)
            metrics = compute_metrics(y_true, preds, label=model_name)
            results[model_name] = metrics["macro_f1"]
            # Free GPU memory
            del mdl; torch.cuda.empty_cache() if torch.cuda.is_available() else None
        except Exception as exc:
            print(f"  ✗ {model_name} failed: {exc}")
            results[model_name] = 0.0

    print(f"\n{'─'*60}")
    print("  LLM Comparison Summary:")
    for name, score in sorted(results.items(), key=lambda x: -x[1]):
        print(f"    {name:35s}  Macro-F1 = {score:.4f}")
    best = max(results, key=results.get)
    print(f"\n  ✓ Best LLM selected: {best}  (Macro-F1 = {results[best]:.4f})")
    return best


### `train_unsup` — Train Unsupervised (LLM) Model

In [ ]:
def train_unsup(train_file: str, val_file: str, model_dir: str,
                best_llm: str = None,
                skip_llm_comparison: bool = False) -> None:
    """
    Unsupervised training function.

    Steps:
      1. Load and display train / validation data statistics.
      2. Compare candidate LLMs on a validation sample (unless skipped).
      3. Save the selected LLM name to model_dir.
      4. Evaluate all 5 prompt templates on the full validation set using the
         best LLM, printing metrics for each.

    Args:
        train_file          : Path to training CSV.
        val_file            : Path to validation CSV.
        model_dir           : Directory to save model artefacts.
        best_llm            : If provided, skip comparison and use this model.
        skip_llm_comparison : If True, use best_llm directly (faster).
    """
    print("\n" + "="*60)
    print("  TRAIN UNSUPERVISED (LLM Prompting)")
    print("="*60)

    train_df = read_data(train_file)
    val_df   = read_data(val_file)
    print_dataset_statistics(train_df, "Train")
    print_dataset_statistics(val_df,   "Validation")

    os.makedirs(model_dir, exist_ok=True)

    # ── Step 1: LLM Selection ────────────────────────────────────────────────
    if skip_llm_comparison and best_llm:
        selected_llm = best_llm
        print(f"  LLM comparison skipped. Using: {selected_llm}")
    else:
        selected_llm = compare_llms(val_df, sample_size=150, prompt_key="Prompt_1")

    # Save selected LLM name
    llm_path = os.path.join(model_dir, "best_llm.txt")
    with open(llm_path, "w") as f:
        f.write(selected_llm)
    print(f"  Best LLM saved → {llm_path}")

    # ── Step 2: Prompt Evaluation on Full Validation Set ─────────────────────
    print(f"\n{'='*60}")
    print(f"  Prompt Evaluation using {selected_llm}")
    print(f"{'='*60}")

    set_global_seed(STUDENT_ID)
    device = get_device()
    tok = AutoTokenizer.from_pretrained(selected_llm)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(selected_llm).to(device)

    y_true    = val_df["sentiment"].tolist()
    texts     = val_df["text"].tolist()
    bs        = 16 if device.type == "cuda" else 8
    prompt_results = {}

    for pkey, template in PROMPTS.items():
        print(f"\n  Evaluating {pkey} …")
        preds   = batch_predict(mdl, tok, device, texts, template, batch_size=bs)
        metrics = compute_metrics(y_true, preds, label=pkey)
        prompt_results[pkey] = metrics

    # Summary table
    print(f"\n{'='*60}")
    print(f"  Prompt Evaluation Summary (Val set, n={len(val_df)})")
    print(f"  {'Prompt':<15} {'Accuracy':>10} {'Macro-F1':>10} {'Wtd-F1':>10}")
    print(f"  {'─'*15} {'─'*10} {'─'*10} {'─'*10}")
    for pkey, m in prompt_results.items():
        print(f"  {pkey:<15} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f} {m['weighted_f1']:>10.4f}")

    best_prompt = max(prompt_results, key=lambda k: prompt_results[k]["macro_f1"])
    print(f"\n  ✓ Best prompt: {best_prompt} "
          f"(Macro-F1 = {prompt_results[best_prompt]['macro_f1']:.4f})")

    # Persist prompt results
    save_object(prompt_results, os.path.join(model_dir, "prompt_results.sav"))
    print("\n  ✓ train_unsup complete.")


### `test_unsup` — Test Unsupervised Model

In [ ]:
def test_unsup(test_file: str, model_dir: str) -> pd.DataFrame:
    """
    Apply all 5 prompts to the test set using the saved best LLM.
    Adds columns out_label_Prompt_1 … out_label_Prompt_5 to test.csv.

    Args:
        test_file : Path to test CSV.
        model_dir : Directory containing saved model artefacts.

    Returns:
        DataFrame with prediction columns added.
    """
    print("\n" + "="*60)
    print("  TEST UNSUPERVISED (LLM Prompting)")
    print("="*60)

    # Load best LLM name (auto-download from Drive if missing)
    llm_path = os.path.join(model_dir, "best_llm.txt")
    if not os.path.exists(llm_path):
        url = GDRIVE_URLS.get("best_llm.txt", "")
        if url and "REPLACE_WITH" not in url:
            print("  Downloading best_llm.txt from Google Drive …")
            os.makedirs(model_dir, exist_ok=True)
            gdown.download(url, llm_path, quiet=False)
        else:
            raise FileNotFoundError(
                f"'{llm_path}' not found. Run train_unsup first "
                "or set GDRIVE_URLS['best_llm.txt']."
            )
    with open(llm_path) as f:
        selected_llm = f.read().strip()
    print(f"  Loaded LLM: {selected_llm}")

    test_df = read_data(test_file)
    texts   = test_df["text"].tolist()

    set_global_seed(STUDENT_ID)
    device = get_device()
    tok = AutoTokenizer.from_pretrained(selected_llm)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(selected_llm).to(device)
    bs  = 16 if device.type == "cuda" else 8

    for pkey, template in PROMPTS.items():
        col = f"out_label_{pkey}"    # e.g. out_label_Prompt_1
        print(f"  Running {pkey} → {col} …")
        preds = batch_predict(mdl, tok, device, texts, template, batch_size=bs)
        test_df[col] = preds
        vc = pd.Series(preds).value_counts()
        print(f"    Distribution: {dict(vc)}")

    # Save output
    out_path = os.path.join(os.path.dirname(test_file), "test.csv")
    test_df.to_csv(out_path, index=False)
    print(f"\n  ✓ Predictions saved → {out_path}")
    print(f"  Columns: {[c for c in test_df.columns if 'out_label' in c]}")
    return test_df


# Method Discriminative — Logistic Regression

This section implements TF-IDF + Logistic Regression with systematic grid search over:
- 4 text pre-processing strategies (basic, stop-word removal, stemming, lemmatisation)
- 2 n-gram ranges ((1,1) and (1,2))
- 3 regularisation values (C ∈ {0.1, 1.0, 10.0})

Total: **24 configurations** evaluated on the validation set.


### Text Pre-processing Functions

In [ ]:
# Shared NLTK objects (instantiated once)
_STOP_WORDS  = set(stopwords.words("english"))
_STEMMER     = PorterStemmer()
_LEMMATIZER  = WordNetLemmatizer()


def preprocess_basic(text: str) -> str:
    """Lowercase only — minimal cleaning."""
    return str(text).lower().strip()


def preprocess_stopwords(text: str) -> str:
    """Lowercase + punctuation removal + stop-word removal."""
    text = str(text).lower().strip()
    tokens = [t for t in text.split() if t.isalpha() and t not in _STOP_WORDS]
    return " ".join(tokens)


def preprocess_stemming(text: str) -> str:
    """Lowercase + stop-word removal + Porter stemming."""
    text = str(text).lower().strip()
    tokens = [
        _STEMMER.stem(t)
        for t in text.split()
        if t.isalpha() and t not in _STOP_WORDS
    ]
    return " ".join(tokens)


def preprocess_lemmatize(text: str) -> str:
    """Lowercase + stop-word removal + WordNet lemmatisation."""
    text = str(text).lower().strip()
    tokens = [
        _LEMMATIZER.lemmatize(t)
        for t in text.split()
        if t.isalpha() and t not in _STOP_WORDS
    ]
    return " ".join(tokens)


# Registry: name → function
PREPROCESS_FNS = {
    "basic"      : preprocess_basic,
    "stopwords"  : preprocess_stopwords,
    "stemming"   : preprocess_stemming,
    "lemmatize"  : preprocess_lemmatize,
}

print(f"✓ {len(PREPROCESS_FNS)} preprocessing functions registered:",
      list(PREPROCESS_FNS.keys()))


### TF-IDF Builder

In [ ]:
def build_tfidf(train_texts: list, ngram_range: tuple = (1, 1),
                max_features: int = 60_000) -> TfidfVectorizer:
    """
    Fit a TF-IDF vectorizer on `train_texts`.
    Uses sublinear TF scaling to reduce the impact of high-frequency terms.
    """
    vec = TfidfVectorizer(
        ngram_range   = ngram_range,
        max_features  = max_features,
        sublinear_tf  = True,   # log(1 + tf) — recommended for text classification
        strip_accents = "unicode",
        analyzer      = "word",
        min_df        = 2,       # ignore very rare terms
    )
    vec.fit(train_texts)
    print(f"  TF-IDF fitted: ngram={ngram_range}, "
          f"vocab_size={len(vec.vocabulary_):,}")
    return vec


### `train_dis` — Train Logistic Regression

In [ ]:
def train_dis(train_file: str, val_file: str, model_dir: str) -> None:
    """
    Discriminative training function.

    Steps:
      1. Load and display train / validation statistics.
      2. Grid search: 4 preprocessing × 2 n-gram ranges × 3 C values = 24 configs.
      3. Evaluate each config on the validation set using Macro-F1.
      4. Retrain best config on the full training set and save artefacts.

    Args:
        train_file : Path to training CSV.
        val_file   : Path to validation CSV.
        model_dir  : Directory to save model artefacts.
    """
    print("\n" + "="*60)
    print("  TRAIN DISCRIMINATIVE (Logistic Regression + TF-IDF)")
    print("="*60)

    train_df = read_data(train_file)
    val_df   = read_data(val_file)
    print_dataset_statistics(train_df, "Train")
    print_dataset_statistics(val_df,   "Validation")

    os.makedirs(model_dir, exist_ok=True)

    X_train_raw = train_df["text"].tolist()
    y_train     = train_df["sentiment"].tolist()
    X_val_raw   = val_df["sentiment"] if "sentiment" not in val_df.columns else None
    y_val       = val_df["sentiment"].tolist()
    X_val_raw   = val_df["text"].tolist()

    # ── Grid Search ──────────────────────────────────────────────────────────
    grid_results = []
    ngram_ranges = [(1, 1), (1, 2)]
    C_values     = [0.1, 1.0, 10.0]

    print(f"\n  Grid search: {len(PREPROCESS_FNS)} preprocessing × "
          f"{len(ngram_ranges)} n-gram × {len(C_values)} C "
          f"= {len(PREPROCESS_FNS)*len(ngram_ranges)*len(C_values)} configs")
    print(f"  {'#':>3}  {'Preprocess':>12}  {'N-gram':>7}  {'C':>6}  "
          f"{'Acc':>8}  {'MacroF1':>9}  {'WtdF1':>8}")
    print(f"  {'─'*3}  {'─'*12}  {'─'*7}  {'─'*6}  "
          f"{'─'*8}  {'─'*9}  {'─'*8}")

    cfg_num = 0
    for prep_name, prep_fn in PREPROCESS_FNS.items():
        X_tr = [prep_fn(t) for t in X_train_raw]
        X_vl = [prep_fn(t) for t in X_val_raw]
        for ngram in ngram_ranges:
            vec = build_tfidf(X_tr, ngram_range=ngram)
            Xtr_vec = vec.transform(X_tr)
            Xvl_vec = vec.transform(X_vl)
            for C in C_values:
                cfg_num += 1
                set_global_seed(STUDENT_ID)
                clf = LogisticRegression(
                    C             = C,
                    max_iter      = 1000,
                    solver        = "lbfgs",
                    class_weight  = "balanced",   # handles 4:1 imbalance
                    random_state  = STUDENT_ID,
                    multi_class   = "auto",
                )
                clf.fit(Xtr_vec, y_train)
                preds   = clf.predict(Xvl_vec)
                acc     = accuracy_score(y_val, preds)
                mf1     = f1_score(y_val, preds, average="macro",    zero_division=0)
                wf1     = f1_score(y_val, preds, average="weighted", zero_division=0)
                grid_results.append({
                    "config"    : cfg_num,
                    "preprocess": prep_name,
                    "ngram"     : ngram,
                    "C"         : C,
                    "accuracy"  : acc,
                    "macro_f1"  : mf1,
                    "wtd_f1"    : wf1,
                })
                print(f"  {cfg_num:>3}  {prep_name:>12}  {str(ngram):>7}  "
                      f"{C:>6.1f}  {acc:>8.4f}  {mf1:>9.4f}  {wf1:>8.4f}")

    # ── Best configuration ────────────────────────────────────────────────────
    best = max(grid_results, key=lambda r: r["macro_f1"])
    print(f"\n  ✓ Best config → Preprocess={best['preprocess']}, "
          f"N-gram={best['ngram']}, C={best['C']}, "
          f"Val Macro-F1={best['macro_f1']:.4f}")

    # ── Retrain best on full train set ────────────────────────────────────────
    prep_fn   = PREPROCESS_FNS[best["preprocess"]]
    X_tr_best = [prep_fn(t) for t in X_train_raw]
    X_vl_best = [prep_fn(t) for t in X_val_raw]

    best_vec = build_tfidf(X_tr_best, ngram_range=best["ngram"])
    Xtr_best = best_vec.transform(X_tr_best)
    Xvl_best = best_vec.transform(X_vl_best)

    set_global_seed(STUDENT_ID)
    best_clf = LogisticRegression(
        C             = best["C"],
        max_iter      = 1000,
        solver        = "lbfgs",
        class_weight  = "balanced",
        random_state  = STUDENT_ID,
        multi_class   = "auto",
    )
    best_clf.fit(Xtr_best, y_train)

    print("\n  Final model — Validation set performance:")
    val_preds = best_clf.predict(Xvl_best)
    compute_metrics(y_val, val_preds, label="Best LR (Val)")

    # ── Save artefacts ────────────────────────────────────────────────────────
    save_object(best_clf,      os.path.join(model_dir, "lr_model.sav"))
    save_object(best_vec,      os.path.join(model_dir, "lr_vectorizer.sav"))
    save_object(best,          os.path.join(model_dir, "lr_config.sav"))
    save_object(grid_results,  os.path.join(model_dir, "lr_grid_results.sav"))

    print("\n  ✓ train_dis complete.")


### `test_dis` — Test Logistic Regression

In [ ]:
def test_dis(test_file: str, model_dir: str) -> pd.DataFrame:
    """
    Load saved LR artefacts (auto-download if missing) and predict on test set.
    Adds `out_label_LR` column to test.csv.

    Args:
        test_file : Path to test CSV.
        model_dir : Directory containing saved model artefacts.

    Returns:
        DataFrame with out_label_LR column added.
    """
    print("\n" + "="*60)
    print("  TEST DISCRIMINATIVE (Logistic Regression)")
    print("="*60)

    # Load artefacts (with auto-download support)
    clf  = load_object(os.path.join(model_dir, "lr_model.sav"),
                       GDRIVE_URLS.get("lr_model.sav"))
    vec  = load_object(os.path.join(model_dir, "lr_vectorizer.sav"),
                       GDRIVE_URLS.get("lr_vectorizer.sav"))
    cfg  = load_object(os.path.join(model_dir, "lr_config.sav"),
                       GDRIVE_URLS.get("lr_config.sav"))

    print(f"  Loaded LR model: preprocess={cfg['preprocess']}, "
          f"ngram={cfg['ngram']}, C={cfg['C']}")

    test_df  = read_data(test_file)
    prep_fn  = PREPROCESS_FNS[cfg["preprocess"]]
    X_test   = [prep_fn(t) for t in test_df["text"].tolist()]
    X_vec    = vec.transform(X_test)
    preds    = clf.predict(X_vec)

    test_df["out_label_LR"] = preds
    vc = pd.Series(preds).value_counts()
    print(f"  Predictions distribution: {dict(vc)}")

    out_path = os.path.join(os.path.dirname(test_file), "test.csv")
    test_df.to_csv(out_path, index=False)
    print(f"  ✓ Predictions saved → {out_path}")
    return test_df


# Main Execution

Set the flags below to control which steps run.  
For a fresh run, set all flags to `True`.  
For submission evaluation, the auto-download will fetch models if not present locally.


In [ ]:
# ─── Execution flags ────────────────────────────────────────────────────────
RUN_TRAIN_DIS   = True   # Train Logistic Regression (grid search)
RUN_TEST_DIS    = True   # Test  Logistic Regression → out_label_LR
RUN_TRAIN_UNSUP = True   # Train LLM (compare LLMs, evaluate 5 prompts)
RUN_TEST_UNSUP  = True   # Test  LLM → out_label_Prompt_1 … _5
# ─────────────────────────────────────────────────────────────────────────────

# ── LR ─────────────────────────────────────────────────────────────────────
if RUN_TRAIN_DIS:
    train_dis(TRAIN_FILE, VAL_FILE, MODEL_DIS_DIR)

if RUN_TEST_DIS:
    test_dis(TEST_FILE, MODEL_DIS_DIR)

# ── LLM ────────────────────────────────────────────────────────────────────
if RUN_TRAIN_UNSUP:
    # Set skip_llm_comparison=True and supply best_llm to skip the slow
    # multi-model comparison (useful after you have already run it once).
    train_unsup(TRAIN_FILE, VAL_FILE, MODEL_UNSUP_DIR,
                skip_llm_comparison=False)

if RUN_TEST_UNSUP:
    test_unsup(TEST_FILE, MODEL_UNSUP_DIR)


## Output Verification
Confirm all required columns are present in `test.csv`.

In [ ]:
# ─── Verify final test.csv ───────────────────────────────────────────────────
out_path = os.path.join(DATA_PATH, "test.csv")
final_df = pd.read_csv(out_path)

required_cols = [
    "out_label_LR",
    "out_label_Prompt_1",
    "out_label_Prompt_2",
    "out_label_Prompt_3",
    "out_label_Prompt_4",
    "out_label_Prompt_5",
]

print(f"\n{'='*60}")
print("  test.csv Final Column Check")
print(f"{'='*60}")
all_ok = True
for col in required_cols:
    present = col in final_df.columns
    status  = "✓" if present else "✗ MISSING"
    print(f"  {status}  {col}")
    if not present:
        all_ok = False

if all_ok:
    print("\n  ✓ All required output columns present!")
else:
    print("\n  ✗ Some columns are missing. Check the logs above.")

print(f"\n  Total test rows: {len(final_df):,}")
print("\n  Sample predictions (first 5 rows):")
display_cols = ["text"] + [c for c in required_cols if c in final_df.columns]
print(final_df[display_cols].head(5).to_string())
